In [1]:
from pyspark.sql import SparkSession

spark=(SparkSession.Builder().
    config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0,org.postgresql:postgresql:42.7.3,io.delta:delta-spark_2.12:3.2.0").
    config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

In [2]:
spark

## Delta table test

In [ ]:
schema = StructType([
    StructField("id", IntegerType(), False),
    StructField("name", StringType(), True),
])
test_df = spark.createDataFrame([(1, "mohan"), (2, "Ram"), (3, "Vijay")], schema)

path = "/home/jovyan/lakehouse/bronze/test"

# Write
test_df.write.format("delta").mode("overwrite").save(path)
## default snappy parquet with delta_log jsons


In [10]:

# Read back
result_df = spark.read.format("delta").load(path)
result_df.show()

+---+-----+
| id| name|
+---+-----+
|  1|mohan|
|  3|Vijay|
|  2|  Ram|
+---+-----+



## Spark Batch from Kafka

In [ ]:
# raw serialized data stored in kafka server

raw_df = (
    spark.read.format("kafka")
    .option("kafka.bootstrap.servers", "kafka:29092")
    .option("subscribe", "upi_transactions")
    .option("startingOffsets", "earliest")
    .load()
)

raw_df.show()


+--------------------+--------------------+----------------+---------+------+--------------------+-------------+
|                 key|               value|           topic|partition|offset|           timestamp|timestampType|
+--------------------+--------------------+----------------+---------+------+--------------------+-------------+
|[75 73 65 72 30 3...|[7B 22 74 78 6E 5...|upi_transactions|        0|     0|2026-08-02 06:34:...|            0|
|[75 73 65 72 30 3...|[7B 22 74 78 6E 5...|upi_transactions|        0|     1|2026-08-02 06:34:...|            0|
|[75 73 65 72 30 3...|[7B 22 74 78 6E 5...|upi_transactions|        0|     2|2026-08-02 06:34:...|            0|
|[75 73 65 72 30 3...|[7B 22 74 78 6E 5...|upi_transactions|        0|     3|2026-08-02 06:34:...|            0|
|[75 73 65 72 30 3...|[7B 22 74 78 6E 5...|upi_transactions|        0|     4|2026-08-02 06:34:...|            0|
|[75 73 65 72 30 3...|[7B 22 74 78 6E 5...|upi_transactions|        0|     5|2026-08-02 06:34:..

In [20]:
raw_df.selectExpr("value","cast(value as string) as json_value").show(1)
# casting value (binary) in kafka resolves the actial value

+--------------------+--------------------+
|               value|          json_value|
+--------------------+--------------------+
|[7B 22 74 78 6E 5...|{"txn_id":"8b4e86...|
+--------------------+--------------------+
only showing top 1 row



In [21]:
# class Transaction(BaseModel):
#     model_config = ConfigDict(strict=True)  # no silent coercion

#     txn_id: str
#     timestamp: datetime
#     sender_upi: str
#     sender_state: str
#     sender_device_id: str
#     receiver_upi: str
#     receiver_type: str
#     receiver_category: str | None = None
#     amount: float
#     status: str
#     is_fraud: bool
#     fraud_pattern: str | None = None


# create schema for spark. spark dont know kakfa datatypes so this declaration is needed
# while reading a delta table,  this is not needed because spark can infer from the delta table metadata

from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType, BooleanType,IntegerType

schema=StructType([StructField("txn_id",StringType(),nullable=False),
                   StructField("timestamp",TimestampType(),nullable=False),
                   StructField("sender_upi",StringType(),nullable=False),
                   StructField("sender_state",StringType(),nullable=False),
                   StructField("sender_device_id",StringType(),nullable=False),
                   StructField("receiver_upi",StringType(),nullable=False),
                   StructField("receiver_type",StringType(),nullable=False),
                   StructField("receiver_category",StringType(),nullable=False),
                   StructField("amount",DoubleType(),nullable=False),
                   StructField("status",StringType(),nullable=False),
                   StructField("is_fraud",BooleanType(),nullable=False),
                   StructField("fraud_pattern",StringType(),nullable=False),])


from pyspark.sql.functions import from_json, col

# Step 1: binary -> string
# Kafka's `value` column is `binary`. from_json can't touch bytes directly.
json_strings_df = raw_df.selectExpr(
    "CAST(key AS STRING) AS sender_upi",   # keep the key if you want it
    "CAST(value AS STRING) AS json_str",
    "timestamp AS kafka_timestamp"          # Kafka's own ingestion timestamp
)

# Step 2: string -> nested struct
# from_json takes a column of JSON strings + your StructType, returns
# a new column typed as `struct<...>` matching the schema. One column,
# not many yet.
parsed_df = json_strings_df.withColumn(
    "parsed", from_json(col("json_str"), schema)
)
parsed_df.show(2,0)

+------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|sender_upi        |json_str                                                                                                                                                                                                                                                                                                                                                   |kafka_timestamp        |parsed                                    

In [22]:


# Step 3: struct -> flat columns
# parsed.* expands every field inside the struct to a top-level column.
# You typically drop the raw json_str here since it's now redundant.
flat_df = parsed_df.select(
    "kafka_timestamp",
    "parsed.*"
)

flat_df.show(2,0)
flat_df.printSchema()

+-----------------------+------------------------------------+--------------------------+------------------+------------+----------------+-------------------+-------------+-----------------+-------+-------+--------+-------------+
|kafka_timestamp        |txn_id                              |timestamp                 |sender_upi        |sender_state|sender_device_id|receiver_upi       |receiver_type|receiver_category|amount |status |is_fraud|fraud_pattern|
+-----------------------+------------------------------------+--------------------------+------------------+------------+----------------+-------------------+-------------+-----------------+-------+-------+--------+-------------+
|2026-08-02 06:34:42.725|8b4e86d3-fbad-4450-98d4-3d7d0e48313e|2026-08-02 06:34:42.341331|user0472@okbizaxis|Telangana   |user0472_0      |user0354@oksbi     |P2P          |NULL             |1343.38|SUCCESS|false   |NULL         |
|2026-08-02 06:34:42.965|9f106b23-cca4-417f-8ede-31f005af8836|2026-08-02 06:34:4

In [ ]:
flat_df.write.format("delta").mode("overwrite").save('/home/jovyan/lakehouse/bronze/test2')

In [26]:
spark.read.format("delta").load('/home/jovyan/lakehouse/bronze/test2').show()

+--------------------+--------------------+--------------------+-------------------+-------------+----------------+--------------------+-------------+-----------------+-------+-------+--------+-------------------+
|     kafka_timestamp|              txn_id|           timestamp|         sender_upi| sender_state|sender_device_id|        receiver_upi|receiver_type|receiver_category| amount| status|is_fraud|      fraud_pattern|
+--------------------+--------------------+--------------------+-------------------+-------------+----------------+--------------------+-------------+-----------------+-------+-------+--------+-------------------+
|2026-08-02 06:34:...|8b4e86d3-fbad-445...|2026-08-02 06:34:...| user0472@okbizaxis|    Telangana|      user0472_0|      user0354@oksbi|          P2P|             NULL|1343.38|SUCCESS|   false|               NULL|
|2026-08-02 06:34:...|9f106b23-cca4-417...|2026-08-02 06:34:...| user0304@okbizaxis|   Tamil Nadu|      user0304_1| user0043@okhdfcbank|        

## Spark Stream

In [28]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType, BooleanType,IntegerType

schema=StructType([StructField("txn_id",StringType(),nullable=False),
                   StructField("timestamp",TimestampType(),nullable=False),
                   StructField("sender_upi",StringType(),nullable=False),
                   StructField("sender_state",StringType(),nullable=False),
                   StructField("sender_device_id",StringType(),nullable=False),
                   StructField("receiver_upi",StringType(),nullable=False),
                   StructField("receiver_type",StringType(),nullable=False),
                   StructField("receiver_category",StringType(),nullable=False),
                   StructField("amount",DoubleType(),nullable=False),
                   StructField("status",StringType(),nullable=False),
                   StructField("is_fraud",BooleanType(),nullable=False),
                   StructField("fraud_pattern",StringType(),nullable=False),])


from pyspark.sql.functions import from_json, col


raw_df = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "kafka:29092")
    .option("subscribe", "upi_transactions")
    .option("startingOffsets", "earliest")
    .load()
)

json_strings_df = raw_df.selectExpr(
    "CAST(key AS STRING) AS sender_upi",   # keep the key if you want it
    "CAST(value AS STRING) AS json_str",
    "timestamp AS kafka_timestamp"          # Kafka's own ingestion timestamp
)

# Step 2: string -> nested struct
# from_json takes a column of JSON strings + your StructType, returns
# a new column typed as `struct<...>` matching the schema. One column,
# not many yet.
parsed_df = json_strings_df.withColumn(
    "parsed", from_json(col("json_str"), schema)
)

flat_df = parsed_df.select(
    "kafka_timestamp",
    "parsed.*"
)

In [ ]:
## write stream

# checkpoint tells spark from which data to process and which are already processed and writen to bronze (avoid dupes)
# .start returns a query object triggering a background job

query = (
    flat_df.writeStream
    .format("delta")
    .option("checkpointLocation", "/home/jovyan/lakehouse/bronze/upi_raw_checkpoint")
    .outputMode("append")
    .start("/home/jovyan/lakehouse/bronze/upi_raw")
)
query.status

{'message': 'Initializing sources',
 'isDataAvailable': False,
 'isTriggerActive': False}

In [32]:
query.status

{'message': 'Waiting for data to arrive',
 'isDataAvailable': False,
 'isTriggerActive': False}

In [40]:
query.status

{'message': 'Getting offsets from KafkaV2[Subscribe[upi_transactions]]',
 'isDataAvailable': False,
 'isTriggerActive': True}

In [ ]:
query.lastProgress


{'id': '54edcc85-6b83-42d6-9910-24dbcdec93b7',
 'runId': 'f4cbfd62-85f0-427b-9c05-74465748ffab',
 'name': None,
 'timestamp': '2026-08-03T20:20:12.194Z',
 'batchId': 1,
 'numInputRows': 0,
 'inputRowsPerSecond': 0.0,
 'processedRowsPerSecond': 0.0,
 'durationMs': {'latestOffset': 2, 'triggerExecution': 2},
 'stateOperators': [],
 'sources': [{'description': 'KafkaV2[Subscribe[upi_transactions]]',
   'startOffset': {'upi_transactions': {'0': 20}},
   'endOffset': {'upi_transactions': {'0': 20}},
   'latestOffset': {'upi_transactions': {'0': 20}},
   'numInputRows': 0,
   'inputRowsPerSecond': 0.0,
   'processedRowsPerSecond': 0.0,
   'metrics': {'avgOffsetsBehindLatest': '0.0',
    'maxOffsetsBehindLatest': '0',
    'minOffsetsBehindLatest': '0'}}],
 'sink': {'description': 'DeltaSink[/home/jovyan/lakehouse/bronze/upi_raw]',
  'numOutputRows': -1}}

>startOffset: {'0': 20}, endOffset: {'0': 20}, latestOffset: {'0': 20}, and avgOffsetsBehindLatest: 0.0. 

That means partition 0 is already sitting at offset 20, and the query is fully caught up — zero offsets behind latest. numInputRows: 0 for this specific batch (batchId 1) just means nothing new arrived between the last check and this one, because everything up to offset 20 was already consumed in an earlier batch

In [42]:
query.exception()

In [43]:
spark.read.format("delta").load("/home/jovyan/lakehouse/bronze/upi_raw").count()
spark.read.format("delta").load("/home/jovyan/lakehouse/bronze/upi_raw").show()

+--------------------+--------------------+--------------------+-------------------+-------------+----------------+--------------------+-------------+-----------------+-------+-------+--------+-------------------+
|     kafka_timestamp|              txn_id|           timestamp|         sender_upi| sender_state|sender_device_id|        receiver_upi|receiver_type|receiver_category| amount| status|is_fraud|      fraud_pattern|
+--------------------+--------------------+--------------------+-------------------+-------------+----------------+--------------------+-------------+-----------------+-------+-------+--------+-------------------+
|2026-08-02 06:34:...|8b4e86d3-fbad-445...|2026-08-02 06:34:...| user0472@okbizaxis|    Telangana|      user0472_0|      user0354@oksbi|          P2P|             NULL|1343.38|SUCCESS|   false|               NULL|
|2026-08-02 06:34:...|9f106b23-cca4-417...|2026-08-02 06:34:...| user0304@okbizaxis|   Tamil Nadu|      user0304_1| user0043@okhdfcbank|        